# nb00 Data Pull: Medi-Cal Managed Care Capitation Rates

**Purpose**
- Download the seven capitation rate CSVs (one per managed care plan model) and the DHCS read me from the CHHS open data portal.
- Save every file unmodified to `data/raw/` with stable names.
- Print structure diagnostics (columns, distinct values, rate formatting) used to design the cleaning step.

**Prerequisites**
- Python 3 with `requests` and `pandas` installed.
- Run this notebook from the `tableau_capitation_rates/notebooks/` folder.
- Internet access to `data.chhs.ca.gov`.

**Outputs**
- `data/raw/raw_two_plan.csv`, `raw_cohs.csv`, `raw_gmc.csv`, `raw_regional.csv`, `raw_single_plan.csv`, `raw_scan.csv`, `raw_pace.csv`
- `data/raw/capitation_rates_read_me.md`
- Diagnostic printout at the end (no files modified).

**Dataset**: [Medi-Cal Managed Care Capitation Rates by Managed Care Plan Models (consolidated)](https://data.chhs.ca.gov/dataset/medi-cal-managed-care-capitation-rates-by-managed-care-plan-models-consolidated), publisher DHCS, coverage 2021 to 2026, updated annually.

In [1]:
# Step 0: mirror all printed output to a text file for easy sharing
import sys
from pathlib import Path

SINK_PATH = Path.cwd() / "nb00_data_pull_cell_output.txt"

# Keep a handle to the notebook's original streams so re-running this cell never double-wraps
_orig_out = getattr(sys, "_nb_orig_stdout", sys.stdout)
_orig_err = getattr(sys, "_nb_orig_stderr", sys.stderr)
sys._nb_orig_stdout, sys._nb_orig_stderr = _orig_out, _orig_err

class _Tee:
    def __init__(self, stream, fh):
        self.stream, self.fh = stream, fh
    def write(self, data):
        self.stream.write(data)
        self.fh.write(data)
        self.fh.flush()
    def flush(self):
        self.stream.flush()
        self.fh.flush()

_sink = open(SINK_PATH, "w")   # a fresh run of this cell starts the file over
sys.stdout = _Tee(_orig_out, _sink)
sys.stderr = _Tee(_orig_err, _sink)
print(f"Mirroring cell output to {SINK_PATH.name} (attach this file in the chat)")

Mirroring cell output to nb00_data_pull_cell_output.txt (attach this file in the chat)
Raw folder: /Users/trinidadcisneros/Documents/Development/trinidadcisneros.github.io/folders/ds_blogs/projects/tableau/tableau_capitation_rates/data/raw
Dataset: Medi-Cal Managed Care Capitation Rates by Managed Care Plan Models
Resources listed: 9

- County Organized Health Systems (COHS) Model | CSV | modified 2026-07-22T17:22:46.707125  <- pulling
- Geographic Managed Care (GMC) | CSV | modified 2026-07-02T20:13:39.229662  <- pulling
- Program of All-Inclusive Care for the Elderly (PACE) Rates | CSV | modified 2025-09-16T22:14:26.968561  <- pulling
- Regional Model/Rural Expansion | CSV | modified 2026-07-02T20:12:03.308532  <- pulling
- Senior Care Action Network (SCAN) | CSV | modified 2026-06-23T22:26:53.055761  <- pulling
- Single Plan Model | CSV | modified 2026-07-02T20:16:31.344972  <- pulling
- Two-Plan Model | CSV | modified 2026-07-02T20:19:13.014175  <- pulling
- Medi-Cal Managed Care C

In [2]:
# Step 1: imports and folder setup
from pathlib import Path
import requests
import pandas as pd

PROJECT_ROOT = Path.cwd().parent          # notebooks/ -> project root
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = "medi-cal-managed-care-capitation-rates-by-managed-care-plan-models-consolidated"
CKAN_API = "https://data.chhs.ca.gov/api/3/action/package_show"

# Stable local name for each CHHS resource id (resource ids are permanent; filenames change yearly)
RESOURCES = {
    "833ddd94-d5ea-4bbc-8980-ba236ab64bf8": "raw_two_plan.csv",
    "b053161a-8155-43e5-98ab-11274ecd9cda": "raw_cohs.csv",
    "7d140e72-4dc6-4c5f-88b2-22e96a9876e1": "raw_gmc.csv",
    "d55907d0-ffbe-4817-9d08-74ff2c85e74a": "raw_regional.csv",
    "1d96751e-59c3-4cd2-9c75-32a6e777ba4b": "raw_single_plan.csv",
    "e9e46e92-b9cd-4cef-bd0f-13448e3e6a9a": "raw_scan.csv",
    "e2af0f7c-2b53-46a5-8bf0-601e2e51f3d1": "raw_pace.csv",
    "2950026a-b62e-4048-9758-3aa97a89ab13": "capitation_rates_read_me.md",
}

print(f"Raw folder: {RAW_DIR}")

In [3]:
# Step 2: resolve the current download URL for each resource via the CKAN API
resp = requests.get(CKAN_API, params={"id": DATASET_ID}, timeout=60)
resp.raise_for_status()
pkg = resp.json()["result"]

url_by_id = {}
print(f"Dataset: {pkg['title']}")
print(f"Resources listed: {len(pkg['resources'])}\n")
for r in pkg["resources"]:
    url_by_id[r["id"]] = r["url"]
    marker = "  <- pulling" if r["id"] in RESOURCES else "  (skipped)"
    print(f"- {r['name']} | {r['format']} | modified {r.get('last_modified')}{marker}")

missing = [rid for rid in RESOURCES if rid not in url_by_id]
assert not missing, f"Resource ids not found in dataset (check on the CHHS page): {missing}"

In [4]:
# Step 3: download every file unmodified to data/raw/
session = requests.Session()
session.headers.update({"User-Agent": "trinidadcisneros.com data story pipeline"})

for rid, local_name in RESOURCES.items():
    out_path = RAW_DIR / local_name
    r = session.get(url_by_id[rid], timeout=120, allow_redirects=True)
    r.raise_for_status()
    out_path.write_bytes(r.content)
    print(f"saved {local_name:32s} {out_path.stat().st_size:>10,} bytes")

In [5]:
# Step 4: diagnostics, part 1. Shape and columns of every model file
csv_files = [n for n in RESOURCES.values() if n.endswith(".csv")]
frames = {}
for name in csv_files:
    df = pd.read_csv(RAW_DIR / name, dtype=str)
    frames[name] = df
    print(f"\n=== {name} === shape {df.shape}")
    print("columns:", list(df.columns))
    print(df.head(3).to_string(index=False))

In [6]:
# Step 5: diagnostics, part 2. Distinct values that drive the cleaning and Tableau design
for name, df in frames.items():
    print(f"\n=== {name} ===")
    for col in df.columns:
        nun = df[col].nunique(dropna=True)
        if nun <= 40:
            print(f"{col} ({nun}): {sorted(df[col].dropna().unique().tolist())}")
        else:
            print(f"{col} ({nun} distinct): sample {df[col].dropna().unique()[:8].tolist()}")

In [7]:
# Step 6: diagnostics, part 3. Los Angeles rows in the Two-Plan file, and rate value formatting
tp = frames["raw_two_plan.csv"]
county_col = [c for c in tp.columns if "county" in c.lower()][0]
la = tp[tp[county_col].str.contains("Los Angeles", case=False, na=False)]
print(f"Los Angeles rows in Two-Plan file: {len(la)}")
print(la.head(20).to_string(index=False))

rate_cols = [c for c in tp.columns if c.lower() in ("lower bound", "midpoint", "upper bound")]
print("\nRate formatting samples:")
for c in rate_cols:
    print(f"{c}: {tp[c].dropna().unique()[:6].tolist()}")

In [8]:
# Final step: confirm the output sink
sys.stdout.flush()
print(f"\nAll printed output saved to: {SINK_PATH}")
print(f"File size: {SINK_PATH.stat().st_size:,} bytes")

**Next step**
- Send the full printed output of Steps 2 through 6 back to the chat.
- The cleaning step (nb01) gets written from those diagnostics: standardizing columns, converting rate text to numbers, and keeping each model as its own clean CSV so the union and all joins happen inside Tableau.
- All printed output is mirrored to `nb00_data_pull_cell_output.txt` in this folder; attach that file in the chat instead of copying cell output. If a cell errors with a red traceback, copy that traceback separately, since the mirror captures printed output only.